In [1]:
import os
import sys 
os.chdir("/workspaces/dev/app")
sys.path.append("/workspaces/dev/app")

In [2]:
import pathlib
import textwrap
import google.generativeai as genai
from IPython.display import display
from IPython.display import Markdown
from dotenv import load_dotenv

/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
load_dotenv()

True

In [4]:
GOOGLE_API_KEY=os.getenv("GOOGLE_API_KEY")

In [5]:
genai.configure(api_key=GOOGLE_API_KEY)

In [6]:
model = genai.GenerativeModel(
  'models/gemini-2.0-flash-lite',
  generation_config=genai.types.GenerationConfig(
    temperature=0.0,
    max_output_tokens=1024,
  )
)

In [7]:
def to_markdown(text):
  text = text.replace('•', '  *')
  return Markdown(textwrap.indent(text, '> ', predicate=lambda _: True))

In [8]:
import re

def extract_tagged_output(text: str):
    context_match = re.search(r"<context>(.*?)</context>", text, re.DOTALL)
    agenda_match = re.search(r"<agenda>(.*?)</agenda>", text, re.DOTALL)

    context = context_match.group(1).strip() if context_match else None
    agenda_raw = agenda_match.group(1).strip() if agenda_match else ""
    agenda = [int(a) for a in agenda_raw.split(",")] if agenda_raw else []

    return context, agenda

In [9]:
def print_tagged_output(cnt_msg:str, response: str, agenda:list[str]):
  print("현재 메시지" + "-" * 40)
  print(cnt_msg)
  print("응답" + "-" * 10)
  print(response)

  context, agenda_ = extract_tagged_output(response)
  if context:
    print("Context:" + "-" * 10)
    print(context)
  else:
    print("No context found.")
  
  if agenda_:
    print("\nAgenda:" + "-" * 10)
    print(agenda_)
    agenda = [a for i, a in enumerate(agenda) if i not in agenda_]
  else:
    print("No agenda found.")
  return agenda

In [10]:
def run_chat(conversation:list[str], prompt:str, last_prompt:str, agenda:list[str], num_people:int, meeting_topic:str):
  chat = model.start_chat(history=[])

  cnt_msg = ""
  for message in conversation:
    cnt_msg += message + "\n"
    if len(cnt_msg) > 500:
      agenda_text = ""
      for i, a in enumerate(agenda):
        agenda_text += f"{i}: {a}\n"
      cnt_msg = prompt.format(
        num_people=num_people,
        meeting_topic=meeting_topic,
        meeting_status="회의 중",
        agenda=agenda_text,
        conversation=cnt_msg
      )
      response = chat.send_message(cnt_msg)
      agenda = print_tagged_output(cnt_msg, response.text, agenda)
      cnt_msg = ""

  if cnt_msg:
      agenda_text = ""
      for i, a in enumerate(agenda):
        agenda_text += f"{i}: {a}\n"
      cnt_msg = prompt.format(
        num_people=num_people,
        meeting_topic=meeting_topic,
        meeting_status="회의 중",
        agenda=agenda_text,
        conversation=cnt_msg
      )
      response = chat.send_message(cnt_msg)
      print_tagged_output(cnt_msg, response.text, agenda)

  response = chat.send_message(last_prompt)
  print(response.text)
  print("-"*40)

In [11]:
prompt = (
"""
너는 회의 내용을 정리하고 분석하는 뉴스 아나운서야.  
사전 지식을 참고하여 지시에 따라 회의 내용을 요약하고, 아젠다 항목을 판별하며, 필요한 경우 행동을 지적해줘.

[사전 지식]
- 회의 인원: {num_people}명
- 회의 주제: {meeting_topic}
- 회의 상태: {meeting_status}

[요약 지침]
- 지금까지의 모든 발화 내용을 참고해도 되지만, **이번 발화 내용에 중심을 두고** 핵심을 한 문장으로 요약해.
- 회의의 주요 논의나 결정을 반복적으로 언급하지 않도록 하고, 발화 내용에 맞게 요약해.
- 말투는 뉴스 보도처럼 객관적이고 간결하게 작성하며, **회의 진행의 흐름에 맞는 핵심을 정확하게 반영**해.
- 문장은 현재 회의 상태에 맞게 작성해줘.
- 마크다운, 리스트, 특수기호 없이 일반 텍스트로 작성하고, 반드시 다음 형식으로 출력해:  
  <context>요약 내용</context>

[아젠다 판별 지침]
- 아래 주어진 아젠다 항목 목록만을 기준으로 판단해.  
- 과거에 주어진 아젠다 항목이나 과거 상태는 고려하지 마.
- 이번 발화 내용을 포함해 과거 발화 흐름을 참고하여, 아래 **현재 아젠다 항목 중 완료된 항목 번호만** 판단해.
- “완료”란 실제로 실행되었거나, 책임자 또는 계획이 명확히 언급된 경우를 의미해.
- 단순한 언급이나 논의는 완료로 간주하지 마.
- 완료된 항목이 없다면 빈 태그로 출력해.
- 반드시 다음 형식으로 출력해:  
  <agenda>1, 3</agenda>

[행동 지적 지침]
- 이번 발화와 직전 발화를 참고하여, 발화자 중 회의 흐름에 부적절하거나 방해가 되는 행동이 드러난 경우, 그 부분을 한 문장으로 지적해.
- 긍정적인 평가나 칭찬은 절대 포함하지 마.
- 문제점이 없는 발화자는 출력하지 마.
- 반드시 다음 형식으로 출력해:
  <correction name="발화자 이름">지적 내용</correction>

[현재 아젠다 항목 목록]
{agenda}

[회의 내용]
{conversation}
"""
)

last_prompt = (
"""
너는 전체 회의 내용을 정리해주는 뉴스 아나운서야.

[요약 지침]
- 지금까지 나눈 회의 대화를 읽고, 핵심 내용을 요약해.
- 전체 흐름에 따라 주제별로 자연스럽게 구성해. 100문장을 넘기지 마.
- 말투는 뉴스 보도처럼 객관적이고 간결하게 유지해.
- 마크다운, 리스트, 기호 없이 일반 텍스트로 작성해.
- 반드시 다음 형식으로 출력해:
  <context>요약 내용</context>
"""
)


In [12]:
text = None
with open("/workspaces/dev/.data/meeting_sample.txt", "r") as f:
  text = f.readlines()

In [13]:
agenda = [
  "회의 목적 설명",
  "프로젝트 진행 상황 공유",
  "이슈 및 장애물 논의",
  "다음 회의 일정 조율"
]


In [14]:
run_chat(
  conversation=text,
  prompt=prompt,
  last_prompt=last_prompt,
  agenda=agenda,
  num_people = "4",
  meeting_topic="미정"
)

현재 메시지----------------------------------------

너는 회의 내용을 정리하고 분석하는 뉴스 아나운서야.  
사전 지식을 참고하여 지시에 따라 회의 내용을 요약하고, 아젠다 항목을 판별하며, 필요한 경우 행동을 지적해줘.

[사전 지식]
- 회의 인원: 4명
- 회의 주제: 미정
- 회의 상태: 회의 중

[요약 지침]
- 지금까지의 모든 발화 내용을 참고해도 되지만, **이번 발화 내용에 중심을 두고** 핵심을 한 문장으로 요약해.
- 회의의 주요 논의나 결정을 반복적으로 언급하지 않도록 하고, 발화 내용에 맞게 요약해.
- 말투는 뉴스 보도처럼 객관적이고 간결하게 작성하며, **회의 진행의 흐름에 맞는 핵심을 정확하게 반영**해.
- 문장은 현재 회의 상태에 맞게 작성해줘.
- 마크다운, 리스트, 특수기호 없이 일반 텍스트로 작성하고, 반드시 다음 형식으로 출력해:  
  <context>요약 내용</context>

[아젠다 판별 지침]
- 아래 주어진 아젠다 항목 목록만을 기준으로 판단해.  
- 과거에 주어진 아젠다 항목이나 과거 상태는 고려하지 마.
- 이번 발화 내용을 포함해 과거 발화 흐름을 참고하여, 아래 **현재 아젠다 항목 중 완료된 항목 번호만** 판단해.
- “완료”란 실제로 실행되었거나, 책임자 또는 계획이 명확히 언급된 경우를 의미해.
- 단순한 언급이나 논의는 완료로 간주하지 마.
- 완료된 항목이 없다면 빈 태그로 출력해.
- 반드시 다음 형식으로 출력해:  
  <agenda>1, 3</agenda>

[행동 지적 지침]
- 이번 발화와 직전 발화를 참고하여, 발화자 중 회의 흐름에 부적절하거나 방해가 되는 행동이 드러난 경우, 그 부분을 한 문장으로 지적해.
- 긍정적인 평가나 칭찬은 절대 포함하지 마.
- 문제점이 없는 발화자는 출력하지 마.
- 반드시 다음 형식으로 출력해:
  <correction name="발화자 이름">지적 내용</correction>

[현재 아젠다 항목 목록]
0: